## 9. Diferenças finitas para EDOs de 2ª ordem

> **Nota de reorganização (rascunho estrutural):** este notebook foi remontado a partir do material do notebook antigo correspondente (pasta `Notebooks_v1_arquivo/`) como parte da reestruturação do curso em 14 notebooks. O conteúdo ainda passará por um ajuste fino (revisão de texto, remoção de redundâncias, exercícios) em uma etapa posterior.

As fórmulas de diferenças finitas para as derivadas de 1ª e 2ª ordem já foram apresentadas no notebook 3. Vamos agora usá-las para resolver **problemas de valor de contorno** de EDOs de 2ª ordem.

### 9.3 Método de diferenças finitas para EDOs de 2ª ordem

Consiste na reformulação do problema contínuo em um problema discreto usando fórmulas de diferenças finitas tomadas sobre uma malha discretizada do domínio do problema.

Vamos apresentar esse método por meio de um exemplo (adaptado de https://www.ufrgs.br/reamat/CalculoNumerico).

**Exemplo 9.3:** Seja o problema de valor de contorno $-y'' = 100(x-1)^2$, $0<x<1$, $y(0)=0$ e $y(1)=0$.

Vamos usar a fórmula de diferenças finitas central de ordem 2 para discretizar a derivada em uma malha uniforme de 11 pontos. 

In [ ]:
xi = np.linspace(0, 1, 11)
print(xi)

A fórmula de diferenças finitas de ordem 2 nos diz que

$$ y''(x_i) \approx \frac{y_{i-1} - 2y_i + y_{i+1}}{h^2}$$

substituindo na equação $-y'' = 100(x-1)^2$, obtemos:

$$-\frac{y_{i-1} - 2y_i + y_{i+1}}{h^2} = 100(x_i-1)^2$$

como $h = 0,1$ podemos escrever 

$$y_{i-1} - 2y_i + y_{i+1} = (x_i-1)^2$$

fazendo $i=1,...,9$ na equação acima, juntamente com as condições de contorno $y_0=0$ e $y_{10}= 0$, obtemos o seguinte sistema linear 11x11.

$$ \begin{cases} 
y_{0} = 0\\
y_{0} -2 y_{1} + y_{2}  = (-0,9)^2\\ 
y_{1} -2 y_{2} + y_{3}  = (-0,8)^2\\ 
y_{2} -2 y_{3} + y_{4}  = (-0,7)^2\\ 
\vdots\\
y_{8} -2 y_{9} + y_{10}  = (-0,1)^2\\ 
y_{10} = 0\\ 
\end{cases} $$


cujas matrizes podem ser obtidas como é mostrado a seguir

In [ ]:
A = np.identity(11)
for i in range(1,10):
    A[i,i-1:i+2] = np.array([1,-2,1])
print(A)

In [ ]:
b = -(xi-1)**2
b[0]=0
b

A solução pode ser obtida fazendo

In [ ]:
yi= np.linalg.solve(A,b)
yi

Esta é a solução numérica do problema de valor de contorno dado. Podemos obter a solução analítica usando SymPy e plotar os fráficos para comparar os resultados.

In [ ]:
import sympy as sp
sp.init_printing()

x = sp.symbols('x')
y = sp.Function('y')(x)
eq = y.diff(x,x)+100*(x-1)**2
sol = sp.dsolve(eq, y, ics={y.subs(x, 0):0, y.subs(x, 1): 0})
sol

In [ ]:
# converte de simbólico para numérico
y_ex = sp.lambdify(x,sol.rhs)
# plota
plt.plot(xi,yi,'.-', xi, y_ex(xi))
plt.grid()

In [ ]:
err = np.mean((yi-y_ex(xi))**2)
print('Erro quadrático médio:',err)

Agora, juntando tudo em um único script, usando um número maior de pontos, obtemos:

In [ ]:
N = 101
h = 1/(N-1)
xi = np.linspace(0, 1, N)
A = np.identity(N)

for i in range(1,N-1):
    A[i,i-1:i+2] = np.array([1,-2,1])

b = -(100*h**2)*(xi-1)**2
b[0]=0

yi= np.linalg.solve(A,b)
y_ex = -25*xi**4/3 + 100*xi**3/3 - 50*xi**2 + 25*xi

err = np.mean((yi-y_ex)**2)
print('Erro quadrático médio (101 pontos):',err)

plt.plot(xi,yi,'r-', lw=1)
plt.plot(xi,y_ex,'b,', )
plt.grid()
plt.show()

**Exemplo 9.4:** Use o método de diferenças finitas para resolver o seguinte problema de valor de contorno:

$$ \begin{cases} 
-y'' + y= e^{-x} \,\text{, } \,\,\,0,5<x<1,5\\
y(0,5)= 1\\ 
y(1,5)= 2
\end{cases} $$

Para tanto, use a fórmula de diferenças finitas central de ordem 2 para discretizar a derivada em uma malha uniforme com passo $h=0,1$. Faça, então, um esboço do gráfico da solução computada.

Substituindo $ y''(x_i)$ por $\frac{y_{i-1}-2y_i+y_{i+1}}{h^2}$, com $h=0,1$, na equação e simplificando obtém-se um sistema linear. 

As equações do sistema serão dadas por 


$$ \begin{cases} 
y_0 = 1\\
-y_0 + 2,01y_1 - y_1= 0,01e^{-0,1}\\ 
-y_1 + 2,01y_2 - y_3= 0,01e^{-0,2}\\ 
-y_2 + 2,01y_3 - y_4= 0,01e^{-0,3}\\ 
\vdots \\
-y_8 + 2,01y_9 + y_{10}= 0,01e^{-0,9}\\
y_{10}= 2
\end{cases} $$

Computacionalmente podemos encontrar a solução seguindo os passos descritos a seguir. Primeiro entramos com os dados do problema.

In [ ]:
N = 11
x0 = 0.5
xn = 1.5
h = (xn-x0)/(N-1)
xi = np.linspace(x0, xn, N)  

Então construímos as matrizes $A$ e $b$.


In [ ]:
A = np.zeros((N,N))  
b = np.zeros(N)  
 
A[0,0] = 1  
b[0] = 1  

for i in np.arange(1,N-1):
    A[i,i-1] = -1  
    A[i,i] = 2+h**2
    A[i,i+1] = -1  
    b[i] = h**2*np.exp(-xi[i])  
A[N-1,N-1] = 1  
b[N-1] = 2  
print(A)

In [ ]:
print(b)

resolvendo o sistema

In [ ]:
yi = np.linalg.solve(A,b)  
print(yi)

Antes de plotar a solução numérica obtida, vamos encontrar a solução analítica usando `sympy`

In [ ]:
x = sp.symbols('x')
y = sp.Function('y')(x)
eq = -y.diff(x,x)+y-sp.exp(-x)
sol = sp.dsolve(eq, y, ics={y.subs(x, 0.5):1, y.subs(x, 1.5): 2})
sol

Calculando o erro quadrático médio

In [ ]:
y_ex =(xi/2 + 0.332107241263667)*np.exp(-xi) + 0.392385373094705*np.exp(xi)

err = np.mean((yi-y_ex)**2)
print('Erro quadrático médio (11 pontos):',err)

Plotando a solução numérica e a solução analítica

In [ ]:
plt.plot(xi,yi,'rx', xi, y_ex,'b--' )
plt.grid()
plt.show()

**Exemplo 9.5:** Considere o seguinte problema de valor de contorno para a equação de calor no estado estacionário 

$$ \begin{cases} 
-y'' = 200e^{-(x-1)^2} \,\text{, } \,\,\,0<x<2\\
y'(0)=0\\ 
y(2)=100
\end{cases} $$

Aproxime a derivada segunda por um esquema de segunda ordem, a derivada primeira na fronteira por um esquema de primeira ordem e transforme a equação diferencial em um sistema de equações lineares. Resolva o sistema linear obtido.

*Solução exata com* `sympy`:

In [ ]:
x = sp.symbols('x')
y = sp.Function('y')(x)
eq = y.diff(x,x) + 200*sp.exp(-(x-1)**2)
sol = sp.dsolve(eq, y, ics={y.diff(x).subs(x, 0):0, y.subs(x, 2): 100})
sol

Observe que a solução não pode ser expressa em termos de funções elementares. Nesse caso, a resolução numérica é a abordagem mais indicada.

*Solução numérica:*

Escolhendo uma discretização em 21 pontos no intervalo de 0 a 2, temos $h=0,2$. O esquema de fiferenças finitas de 1ª ordem na fronteira esquerda será dado pela equação

$$y'(x_i)\approx\ \frac{f(x_{i+1})-f(x_i)}{h}  \,\,\,\,\,\text{ou} \,\,\,\, y'_0=0\approx\frac{y_1-y_0}{0,1}$$

assim as equações fornecidas pelas condições de contorno são

$$ y_1-y_0 = 0 \,\,\,\,\,\text{e} \,\,\,\, y_{20}=100$$

As demais equações serão fornecidas pelo esquema de segunda ordem, ou seja, serão dadas por

$$ y''(x_i)= 200e^{-(xi-1)^2} \approx \frac{y_{i-1}-2y_i+y_{i+1}}{h^2}$$

que leva a 

$$ y_{i-1}-2y_i+y_{i+1}=2e^{-(xi-1)^2}  \,\,\,\,\,\text{para} \,\,\,\, i=1,...,19$$



Temos, portanto

$$ \begin{cases} 
y_0 - y_1 = 0\\
y_1 - 2y_1 + y_1= 2e^{-(0,2-1)^2}\\
y_2 - 2y_2 + y_3= 2e^{-(0,4-1)^2}\\
\vdots \\
y_{18} - 2y_{18} + y_{19}= 2e^{-(1,6-1)^2}\\
y_{19} - 2y_{18} + y_{19}= 2e^{-(1,8-1)^2}\\
y_{20}= 100
\end{cases} $$


In [ ]:
N = 11
x0 = 0.0
xn = 2.0
h = (xn-x0)/(N-1)
xi = np.linspace(x0, xn, N)  
print('xi=',xi)

A = np.zeros((N,N))  
b = np.zeros(N)  
 
# Condições de contorno
A[0,0] = 1
A[0,1] = -1
A[N-1,N-1] = 1
b[0] = 0
b[N-1] = 100

# montando as matrizes A e b
for i in np.arange(1,N-1):
    A[i,i-1] = 1  
    A[i,i] = -2
    A[i,i+1] = 1  
    b[i] = 2*np.exp(-(xi[i]-1)**2)  
print('A=',A)

#resolvendo o sistema
yi = np.linalg.solve(A,b)  
print('yi=',yi)

#plotando
plt.plot(xi,yi)
plt.grid()
plt.show()

**Exemplo 9.6:** ([Burden, Richard, L. et al., 2016](https://integrada.minhabiblioteca.com.br/reader/books/9788522123414/pageid/783)) Dado o problema de contorno de segunda ordem linear

$$
y^{\prime \prime}=p(x) y^{\prime}+q(x) y+r(x), \quad \text { para } a \leq x \leq b \operatorname{com} y(a)=\alpha \text { e } y(b)=\beta,
$$

a resolução pelo método das diferenças finitas requer aproximações numéricas para $y'$ e $y''$ em um conjunto pontos que dividem o intervalo $[a,b]$ em $N+1$ subintervalos cujos extremos são os pontos $x_i=a+ih$, com $i=0,1,2,...,N+1$ da malha, e $h=(b-a)/(N+1))$.

Aproximando $y'$ e $y''$ pela fórmula de diferenças centradas obtemos uma equação na forma

$$
\left(\frac{-y_{i+1}+2 y_i-y_{i-1}}{h^2}\right)+p\left(x_i\right)\left(\frac{y_{i+1}-y_{i-1}}{2 h}\right)+q\left(x_i\right) y_i=-r\left(x_i\right)
$$

para cada $i=1,2,...,N$. 

As equações podem ser reescritas na forma 
$$
-\left(1+\frac{h}{2} p\left(x_i\right)\right) y_{i-1}+\left(2+h^2 q\left(x_i\right)\right) y_i-\left(1-\frac{h}{2} p\left(x_i\right)\right) y_{i+1}=-h^2 r\left(x_i\right),
$$

que vai gerar o sistema linear tridiagonal $N \times N$ cuja solução dará a solução numérica do problema de valor de contorno.  

Use método apresentado para obter uma aproximação para a solução do problema de contorno

$$
y^{\prime \prime}=-\frac{2}{x} y^{\prime}+\frac{2}{x^2} y+\frac{\operatorname{sen}(\ln x)}{x^2}, \quad \text { para } 1 \leq x \leq 2, \operatorname{com} y(1)=1 \text { e } y(2)=2.
$$


In [ ]:
a = 1.0
b = 2.0
ya = 1
yb = 2

N = 9
h = (b-a)/(N+1)
A = np.zeros((N,N))
b = np.zeros(N)

p = lambda x: -2./x
q = lambda x: 2./x**2
r = lambda x: np.sin(np.log(x))/x**2

In [ ]:
x = a+h
A[0,0] = 2+h**2*q(x)
A[0,1] = -1+(h/2)*p(x)
b[0] = -h**2*r(x)+(1+(h/2)*p(x))*ya
#print(x)

for i in range(1,N-1):
    x = a+(i+1)*h
    A[i,i-1] = -1-(h/2)*p(x)
    A[i,i] = 2+h**2*q(x) 
    A[i,i+1] = -1+(h/2)*p(x)
    b[i] = -h**2*r(x)
    #print(x)
    
x = a+(N)*h
#print (x)
A[N-1,N-2] = -1-(h/2)*p(x)
A[N-1,N-1] = 2+h**2*q(x)
b[N-1] = -h**2*r(x)+(1-(h/2)*p(x))*yb

In [ ]:
print (np.round(A,3))
print (np.round(b,3))

In [ ]:
np.linalg.solve(A,b)

### Exercícios:

(Adaptados de https://www.ufrgs.br/reamat/CalculoNumerico)**:**

**1.** Considere o seguinte problema de valor de contorno para a equação de calor no estado estacionário 

$$ \begin{cases} 
-y'' = 200e^{-(x-1)^2} \,\text{, } \,\,\,0<x<2\\
y(0)=120\\ 
y(2)=100
\end{cases} $$

Aproxime a derivada segunda por um esquema de segunda ordem, a derivada primeira na fronteira por um esquema de primeira ordem e transforme a equação diferencial em um sistema de equações lineares. Resolva o sistema linear obtido. Use uma discretização com 21 pontos.

**2.** Considere o seguinte problema de valor de contorno para a equação de calor no estado estacionário com um termo não linear de radiação

$$-y'' = 100 - \frac{y^4}{10000} \,\text{, } \,\,\,0<x<2$$
$$y(0)=0 \,\,\,\text{e}\,\,\, y(2)=10$$

Aproxime a derivada segunda por um esquema de segunda ordem, a derivada primeira na fronteira por um esquema de primeira ordem e transforme a equação diferencial em um sistema de equações lineares. Resolva o sistema linear obtido.

**3.** Considere o seguinte problema de valor de contorno para a equação de calor no estado estacionário com um termo não linear de radiação e um termo de convecção

$$-y'' +3y'= 100 - \frac{y^4}{10000} \,\text{, } \,\,\,0<x<2$$
$$y'(0)=0 \,\,\,\text{e}\,\,\, y(2)=10$$

Aproxime a derivada segunda por um esquema de segunda ordem, a derivada primeira na fronteira por um esquema de primeira ordem e transforme a equação diferencial em um sistema de equações lineares. Resolva o sistema linear obtido.

**4.** O problema de contorno ([fonte](https://integrada.minhabiblioteca.com.br/reader/books/9788522123414/pageid/789))

$$
y^{\prime \prime}=y^{\prime}+2 y+\cos x, \quad 0 \leq x \leq \frac{\pi}{2}, \quad y(0)=-0,3, \quad y\left(\frac{\pi}{2}\right)=-0,1,
$$

tem a solução $y(x)=-\frac{1}{10}(\operatorname{sen} x+3 \cos x)$. Use o método das diferenças finitas linear para obter uma aproximação para a solução e compare os resultados com a solução real. Repita para diferentes valores de $h$.

**5.** O problema de contorno ([fonte](https://integrada.minhabiblioteca.com.br/reader/books/9788522123414/pageid/789))

$$
y^{\prime \prime}=4(y-x), \quad 0 \leq x \leq 1, \quad y(0)=0, \quad y(1)=2
$$

tem a solução $y(x)=e^2\left(e^4-1\right)^{-1}\left(e^{2 x}-e^{-2 x}\right)+x$. Use o método das diferenças finitas linear para obter uma aproximação para a solução e compare os resultados com a solução real. Repita para diferentes valores de $h$.